[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donamvn/thcs-lap-trinh-python/blob/main/05_tim_kiem.ipynb)

> **Trước khi học:** bấm **Tệp → Lưu bản sao vào Drive** (*File → Save a copy in Drive*) để lưu bài làm của em.
> Nếu Colab hỏi *"Sổ tay này không phải do Google tạo"*, bấm **Vẫn chạy** (*Run anyway*).

# Bài 5 - Thuật toán tìm kiếm

**Bài toán:** cho danh sách `a` và giá trị `x`. Hỏi `x` có trong `a` không, nếu có thì ở vị trí nào?

Ta học hai thuật toán và so sánh xem cái nào **nhanh hơn**. Đây là câu hỏi quan trọng nhất của môn giải thuật.

In [ ]:
# Ô CHUẨN BỊ - em chỉ cần chạy ô này (Shift + Enter), chưa cần hiểu hết.
def kiem_tra(ham, cac_truong_hop):
    """Chạy hàm của em với nhiều đầu vào và so sánh với kết quả mong đợi."""
    dung = 0
    for dau_vao, mong_doi in cac_truong_hop:
        if not isinstance(dau_vao, tuple):
            dau_vao = (dau_vao,)
        tham_so = ", ".join(repr(x) for x in dau_vao)
        try:
            ket_qua = ham(*dau_vao)
        except Exception as loi:
            ket_qua = f"LỖI: {loi}"
        dat = ket_qua == mong_doi
        dung += dat
        print(("  ĐÚNG " if dat else "  SAI  ") + f"{ham.__name__}({tham_so}) = {ket_qua!r}   (mong đợi {mong_doi!r})")
    tong = len(cac_truong_hop)
    print(f"\n=> Em làm đúng {dung}/{tong} trường hợp.", "Tuyệt vời!" if dung == tong else "Thử sửa lại nhé, em làm được mà!")

print("Đã chuẩn bị xong công cụ kiểm tra bài tập.")

## 1. Tìm kiếm tuần tự (tìm lần lượt)

Ý tưởng đơn giản nhất: **xét lần lượt từng phần tử** từ đầu đến cuối, gặp thì dừng.
Giống như tìm một bạn trong hàng bằng cách hỏi từng người: *"Bạn có phải là Chi không?"*

#### Sơ đồ khối: Tìm kiếm tuần tự



<img src="https://raw.githubusercontent.com/donamvn/thcs-lap-trinh-python/main/so-do/05_tuan_tu.png" width="460">

<small>Tải sơ đồ: [ảnh PNG](https://raw.githubusercontent.com/donamvn/thcs-lap-trinh-python/main/so-do/05_tuan_tu.png) · [bản vẽ Excalidraw](https://github.com/donamvn/thcs-lap-trinh-python/blob/main/so-do/05_tuan_tu.excalidraw) · [mã Mermaid](https://github.com/donamvn/thcs-lap-trinh-python/blob/main/so-do/05_tuan_tu.mmd)</small>

In [ ]:
def tim_tuan_tu(a, x):
    for i in range(len(a)):
        print(f"  Bước {i + 1}: xét a[{i}] = {a[i]}")
        if a[i] == x:
            return i
    return -1

lop = ["An", "Bình", "Chi", "Dũng", "Giang", "Hà"]
print("Vị trí của Dũng:", tim_tuan_tu(lop, "Dũng"))
print("Vị trí của Minh:", tim_tuan_tu(lop, "Minh"))

[Xem từng bước trên Python Tutor](https://pythontutor.com/visualize.html#code=def%20tim_tuan_tu%28a%2C%20x%29%3A%0A%20%20%20%20for%20i%20in%20range%28len%28a%29%29%3A%0A%20%20%20%20%20%20%20%20print%28f%22%20%20B%C6%B0%E1%BB%9Bc%20%7Bi%20%2B%201%7D%3A%20x%C3%A9t%20a%5B%7Bi%7D%5D%20%3D%20%7Ba%5Bi%5D%7D%22%29%0A%20%20%20%20%20%20%20%20if%20a%5Bi%5D%20%3D%3D%20x%3A%0A%20%20%20%20%20%20%20%20%20%20%20%20return%20i%0A%20%20%20%20return%20-1%0A%0Alop%20%3D%20%5B%22An%22%2C%20%22B%C3%ACnh%22%2C%20%22Chi%22%2C%20%22D%C5%A9ng%22%2C%20%22Giang%22%2C%20%22H%C3%A0%22%5D%0Aprint%28%22V%E1%BB%8B%20tr%C3%AD%20c%E1%BB%A7a%20D%C5%A9ng%3A%22%2C%20tim_tuan_tu%28lop%2C%20%22D%C5%A9ng%22%29%29%0Aprint%28%22V%E1%BB%8B%20tr%C3%AD%20c%E1%BB%A7a%20Minh%3A%22%2C%20tim_tuan_tu%28lop%2C%20%22Minh%22%29%29&cumulative=false&heapPrimitives=nevernest&mode=display&origin=opt-frontend.js&py=3&rawInputLstJSON=%5B%5D&textReferences=false)

**Nhận xét:** nếu danh sách có `n` phần tử, trường hợp **xấu nhất** (không có `x`, hoặc `x` nằm cuối) phải xét **cả `n`** phần tử.
Với 1 triệu phần tử là 1 triệu bước!

## 2. Tìm kiếm nhị phân (chia đôi)

**Trò chơi đoán số:** bạn nghĩ một số từ 1 đến 100, mình đoán, bạn chỉ trả lời *"lớn hơn"* hoặc *"nhỏ hơn"*.
Cách thông minh nhất là **luôn đoán số ở giữa**. Mỗi lần đoán, khoảng tìm kiếm **giảm một nửa**: 100 → 50 → 25 → 13 → 7 → 4 → 2 → 1.
Chỉ tối đa **7 lần** là chắc chắn đoán trúng!

> **Điều kiện bắt buộc:** danh sách phải được **sắp xếp** trước (như từ điển giấy xếp theo vần A, B, C).

#### Sơ đồ khối: Tìm kiếm nhị phân



<img src="https://raw.githubusercontent.com/donamvn/thcs-lap-trinh-python/main/so-do/05_nhi_phan.png" width="460">

<small>Tải sơ đồ: [ảnh PNG](https://raw.githubusercontent.com/donamvn/thcs-lap-trinh-python/main/so-do/05_nhi_phan.png) · [bản vẽ Excalidraw](https://github.com/donamvn/thcs-lap-trinh-python/blob/main/so-do/05_nhi_phan.excalidraw) · [mã Mermaid](https://github.com/donamvn/thcs-lap-trinh-python/blob/main/so-do/05_nhi_phan.mmd)</small>

In [ ]:
def tim_nhi_phan(a, x):
    trai, phai = 0, len(a) - 1
    buoc = 0
    while trai <= phai:
        buoc += 1
        giua = (trai + phai) // 2
        print(f"  Bước {buoc}: xét đoạn [{trai}..{phai}], giữa là a[{giua}] = {a[giua]}")
        if a[giua] == x:
            return giua
        elif a[giua] < x:
            trai = giua + 1
        else:
            phai = giua - 1
    return -1

a = [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]
print("Tìm 23 -> vị trí", tim_nhi_phan(a, 23))
print("Tìm 7  -> vị trí", tim_nhi_phan(a, 7))

[Xem từng bước trên Python Tutor](https://pythontutor.com/visualize.html#code=def%20tim_nhi_phan%28a%2C%20x%29%3A%0A%20%20%20%20trai%2C%20phai%20%3D%200%2C%20len%28a%29%20-%201%0A%20%20%20%20buoc%20%3D%200%0A%20%20%20%20while%20trai%20%3C%3D%20phai%3A%0A%20%20%20%20%20%20%20%20buoc%20%2B%3D%201%0A%20%20%20%20%20%20%20%20giua%20%3D%20%28trai%20%2B%20phai%29%20//%202%0A%20%20%20%20%20%20%20%20print%28f%22%20%20B%C6%B0%E1%BB%9Bc%20%7Bbuoc%7D%3A%20x%C3%A9t%20%C4%91o%E1%BA%A1n%20%5B%7Btrai%7D..%7Bphai%7D%5D%2C%20gi%E1%BB%AFa%20l%C3%A0%20a%5B%7Bgiua%7D%5D%20%3D%20%7Ba%5Bgiua%5D%7D%22%29%0A%20%20%20%20%20%20%20%20if%20a%5Bgiua%5D%20%3D%3D%20x%3A%0A%20%20%20%20%20%20%20%20%20%20%20%20return%20giua%0A%20%20%20%20%20%20%20%20elif%20a%5Bgiua%5D%20%3C%20x%3A%0A%20%20%20%20%20%20%20%20%20%20%20%20trai%20%3D%20giua%20%2B%201%0A%20%20%20%20%20%20%20%20else%3A%0A%20%20%20%20%20%20%20%20%20%20%20%20phai%20%3D%20giua%20-%201%0A%20%20%20%20return%20-1%0A%0Aa%20%3D%20%5B2%2C%205%2C%208%2C%2012%2C%2016%2C%2023%2C%2038%2C%2056%2C%2072%2C%2091%5D%0Aprint%28%22T%C3%ACm%2023%20-%3E%20v%E1%BB%8B%20tr%C3%AD%22%2C%20tim_nhi_phan%28a%2C%2023%29%29%0Aprint%28%22T%C3%ACm%207%20%20-%3E%20v%E1%BB%8B%20tr%C3%AD%22%2C%20tim_nhi_phan%28a%2C%207%29%29&cumulative=false&heapPrimitives=nevernest&mode=display&origin=opt-frontend.js&py=3&rawInputLstJSON=%5B%5D&textReferences=false)

**Máy tính chơi đoán số:** bí mật là số `73`. Xem máy cần bao nhiêu lần đoán:

In [ ]:
so_bi_mat = 73      # em thử đổi thành số bất kỳ từ 1 đến 100
thap, cao = 1, 100
lan = 0
while True:
    lan += 1
    doan = (thap + cao) // 2
    if doan == so_bi_mat:
        print(f"Lần {lan}: đoán {doan} -> ĐÚNG RỒI!")
        break
    elif doan < so_bi_mat:
        print(f"Lần {lan}: đoán {doan} -> lớn hơn nữa")
        thap = doan + 1
    else:
        print(f"Lần {lan}: đoán {doan} -> nhỏ hơn")
        cao = doan - 1

## 3. So sánh tốc độ: tuần tự và nhị phân

Đếm số bước trong **trường hợp xấu nhất** khi danh sách càng ngày càng dài:

In [ ]:
import matplotlib.pyplot as plt

def so_buoc_nhi_phan(n):
    buoc = 0
    while n > 0:
        n //= 2
        buoc += 1
    return buoc

cac_n = [10, 100, 1000, 10_000, 100_000, 1_000_000]
print(f"{'Số phần tử':>12} | {'Tuần tự':>10} | {'Nhị phân':>8}")
for n in cac_n:
    print(f"{n:>12,} | {n:>10,} | {so_buoc_nhi_phan(n):>8}")

ns = list(range(1, 1001))
plt.figure(figsize=(8, 4))
plt.plot(ns, ns, label="Tìm tuần tự (n bước)")
plt.plot(ns, [so_buoc_nhi_phan(n) for n in ns], label="Tìm nhị phân (khoảng log2 n bước)", linewidth=3)
plt.xlabel("Số phần tử của danh sách")
plt.ylabel("Số bước (xấu nhất)")
plt.title("Tìm nhị phân nhanh hơn rất nhiều!")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

Với **1 triệu** phần tử, tìm tuần tự cần tới 1 000 000 bước, còn tìm nhị phân chỉ cần khoảng **20 bước**!
Đó là lý do Google tìm được thông tin trong hàng tỷ trang web chỉ trong nháy mắt: họ dùng những cấu trúc dữ liệu và thuật toán thông minh.

| | Tìm tuần tự | Tìm nhị phân |
|---|---|---|
| Yêu cầu | Không cần gì | Danh sách **đã sắp xếp** |
| Số bước xấu nhất | `n` | khoảng `log2(n)` |
| Dễ viết | Rất dễ | Cần cẩn thận với `trai`, `phai` |

## 4. Bài tập tự luyện

### Bài 5.1 - Đếm số lần xuất hiện

Viết hàm `dem_xuat_hien(a, x)` trả về số lần `x` xuất hiện trong `a` (không dùng `a.count`).

In [ ]:
def dem_xuat_hien(a, x):
    # Viết code của em ở đây
    return 0

# Chạy ô này để kiểm tra bài làm
kiem_tra(dem_xuat_hien, [(([1, 3, 1, 2, 1], 1), 3), (([1, 2, 3], 5), 0), (([], 1), 0), ((['a', 'b', 'a'], 'a'), 2)])

<details>
<summary><b>Bấm để xem lời giải</b> (hãy tự làm trước nhé!)</summary>

```python
def dem_xuat_hien(a, x):
    dem = 0
    for y in a:
        if y == x:
            dem += 1
    return dem
```
</details>

### Bài 5.2 - Vị trí xuất hiện cuối cùng

Viết hàm `tim_cuoi(a, x)` trả về vị trí xuất hiện **cuối cùng** của `x` trong `a`, hoặc `-1` nếu không có.
*Gợi ý:* tìm tuần tự nhưng đi **từ cuối về đầu**.

In [ ]:
def tim_cuoi(a, x):
    # Viết code của em ở đây
    return -1

# Chạy ô này để kiểm tra bài làm
kiem_tra(tim_cuoi, [(([4, 2, 4, 7], 4), 2), (([1, 2, 3], 9), -1), (([5], 5), 0)])

<details>
<summary><b>Bấm để xem lời giải</b> (hãy tự làm trước nhé!)</summary>

```python
def tim_cuoi(a, x):
    for i in range(len(a) - 1, -1, -1):
        if a[i] == x:
            return i
    return -1
```
</details>

### Bài 5.3 - Tự viết tìm kiếm nhị phân

Không nhìn code ở trên, hãy tự viết hàm `nhi_phan(a, x)` (không cần in các bước) theo sơ đồ khối. Trả về vị trí của `x` hoặc `-1`.

In [ ]:
def nhi_phan(a, x):
    # Viết code của em ở đây
    return -1

# Chạy ô này để kiểm tra bài làm
kiem_tra(nhi_phan, [(([1, 3, 5, 7, 9, 11], 7), 3), (([1, 3, 5, 7, 9, 11], 1), 0), (([1, 3, 5, 7, 9, 11], 11), 5), (([1, 3, 5, 7, 9, 11], 4), -1), (([], 4), -1)])

<details>
<summary><b>Bấm để xem lời giải</b> (hãy tự làm trước nhé!)</summary>

```python
def nhi_phan(a, x):
    trai, phai = 0, len(a) - 1
    while trai <= phai:
        giua = (trai + phai) // 2
        if a[giua] == x:
            return giua
        elif a[giua] < x:
            trai = giua + 1
        else:
            phai = giua - 1
    return -1
```
</details>

## Tóm tắt Bài 5
- **Tìm tuần tự**: đơn giản, dùng được mọi lúc, nhưng chậm với danh sách dài (`n` bước).
- **Tìm nhị phân**: chia đôi liên tục, cực nhanh (khoảng `log2 n` bước), nhưng **cần danh sách đã sắp xếp**.
- Cùng một bài toán có thể có nhiều thuật toán. Người lập trình giỏi biết **chọn thuật toán phù hợp**.

Tìm nhị phân cần danh sách đã sắp xếp. Vậy **sắp xếp** thế nào? Mời em sang **Bài 6**!